#### 1-Librerias

## 1.1-Instalacion librerias externas

In [ ]:
#!pip install /kaggle/input/noisereduce/noisereduce-3.0.3-py3-none-any.whl

## 1.2-Importar librerias

In [ ]:
#import numpy as np # linear algebra
import pandas as pd # data processing, CSV file 
import os # creating directories
import shutil # copiar archivos
import random

# cleaning audio
#import noisereduce as nr
from scipy.signal import butter, filtfilt, sosfiltfilt
import librosa

import os, glob, gc, numpy as np
from pathlib import Path
from tqdm import tqdm

import torch.nn as nn
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision import models
from collections import defaultdict
from torchvision import models as tv

import timm

import torch.serialization
torch.serialization.add_safe_globals([np.core.multiarray.scalar])

## 1.3-Rutas y directorios

In [ ]:
def create_folder(path, carpeta):
    folder = os.path.join(path, carpeta)
    os.makedirs(folder, exist_ok=True)
    print(f'[+] Folder "{carpeta}" created.')
    return folder

def get_path(dir, filename):
    return os.path.join(dir, filename)

def clean_folder(path, delete_folder=False):
    if delete_folder:
        shutil.rmtree(path)
        print(f"[+] Carpeta eliminada: {path}")
    else:
        for filename in os.listdir(path):
            file_path = os.path.join(path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.remove(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f"Error eliminando {file_path}: {e}")
        print(f"[+] Contenido eliminado en: {path}")

In [ ]:
#  birdclef en /kaggle/input/birdclef-2025
path_birdclef = '/kaggle/input/birdclef-2025'
path_home = '/kaggle/working'


path_models = create_folder(path_home, "models")

path_train_audio = get_path(path_birdclef, "train_audio")

path_train_soundscapes = get_path(path_birdclef, "train_soundscapes")
path_test_soundscapes = '/kaggle/input/birdclef-2025/test_soundscapes/'

test_soundscapes = [os.path.join(path_test_soundscapes, afile) for afile in sorted(os.listdir
                                          (path_test_soundscapes)) if afile.endswith('.ogg')]

print(path_test_soundscapes)
print(path_train_soundscapes)

path_taxonomy_csv = get_path(path_birdclef, "taxonomy.csv")

path_sample_submission = get_path(path_birdclef, "sample_submission.csv")

path_data = create_folder(path_home, "data")

# 2-Preprocesamiento

## 2.1- Parametros

In [ ]:
### Parametros modificables ###
# Audio / STFT Config
SAMPLE_RATE = 32000 # Tasa de muestreo, formato del dataset
N_FFT = 1024 # Transformada de Fourier
HOP_LENGTH = 320 # nº muestras entre el inicio de ventanas, ver stft
F_MIN = 20 # Frecuencia minima Hz
F_MAX = 16000 # Frecuencia maxima Hz
MIN_SEG_DURATION_SEC = 0.5 # Duracion minima para intervalos

# Mels
N_MELS = 128 # Nº de bandas del espectograma de mel, reducido 128->64 problemas de espacio
MEL_DURATION = 5.0  # Duracion en segundos por segmento

# VAD Recortes
TOP_DB = None #30# Umbral para detectar actividad, <30 = silencio

# Filtros de banda
APPLY_BANDPASS = True #False
LOWCUT = 150 #255 #300  # Frecuencia de corte inferior
HIGHCUT = 15550 #9000 #15950  # Frecuencia de corte superior

# Pipelina Preprocesado
FLAG_PROCESS= False # True si se quiere preprocesar los datos
FLAG_MAP = False # True si se quiere mapear el train csv
FLAG_LOAD_CSV = False# True si se quiere cargar desde un dataset externo index train, val trina, index audio
RNG_SEED = 42 # semilla para muestreo

## 2.2-Fuciones limpieza y preprocesado

In [ ]:
# Funciones

def load_audio(filepath, sr=SAMPLE_RATE):
    # Cargar el archivo de audio
    y, sr = librosa.load(filepath, sr=sr, mono=True, dtype=np.float32, res_type='kaiser_fast')
    return y, sr


def reduce_noise(y, sr):
    # Reducir el ruido
    y_denoised = nr.reduce_noise(y=y, sr=sr)
    return y_denoised


def bandpass_filter(y, sr, lowcut=LOWCUT, highcut=HIGHCUT, order=5):
    if not APPLY_BANDPASS:
        return y
    # Filtrado para eliminar frecuencias irrelevantes (Filtro paso bajo y filtro paso alto)
    nyquist = 0.5 * sr
    low = max(1.0, float(lowcut))
    high = min(float(highcut), nyquist * 0.98)
    if low >= high: # datos invalidos
        return y
    low = lowcut / nyquist
    high = highcut / nyquist
    # Butterworth formato sos y filtrado cero-fase
    sos= butter(order, [low, high], btype='band', output='sos')
    y_filtered = sosfiltfilt(sos, y) # evitar desfase
    return y_filtered


def normalize_audio(y):
    # Normalizacion la señal del audio, par aque todos los audios tengan un volumen similar
    # (ya sea entre -1 y 1)
    y_normalized = librosa.util.normalize(y)
    return y_normalized


def detect_active_segments(y, sr, top_db=TOP_DB):
    if top_db is None:
        return np.array([[0, len(y)]], dtype=int)
    # devuelve array de pares [start, end]
    return librosa.effects.split(y, top_db=top_db)


def is_likely_speech(y_segment, sr):
    # Determinar si un fragmento contiene voz humana
    pitches, _ = librosa.piptrack(y=y_segment, sr=sr)
    pitch_values = pitches[pitches > 0]
    if len(pitch_values) == 0:
        return False
    median_pitch = np.median(pitch_values)
    return 85 < median_pitch < 255  # rango de voz humana

def preprocess_wave(y, sr, use_bp=False, do_norm=True, denoise_fn=None):
    if denoise_fn is not None:
        y = denoise_fn(y=y, sr=sr)
    if use_bp:
        y = bandpass_filter(y, sr)
    if do_norm:
        y = normalize_audio(y)
    return y

In [ ]:
def extract_mel(y_segment, sr,
                n_mels=N_MELS,
                hop_length=HOP_LENGTH,
                duration=MEL_DURATION,
                n_fft=N_FFT,
                fmin=F_MIN,
                fmax=F_MAX):
    """
    - Ajusta el segmento a 'duration' segundos (pad/cut).
    - Calcula Mel con tus hiperparámetros globales.
    - Usa log1p(mel) para estabilidad numérica y mejor comportamiento en training.
    - Devuelve [n_mels, n_frames] en float32.
    """
    # Ajuste exacto de longitud
    samples = int(duration * sr)
    y_fixed = librosa.util.fix_length(y_segment, size=samples)

    # Mel spectrogram (power=2.0 → espectro de potencia)
    mel = librosa.feature.melspectrogram(
        y=y_fixed,
        sr=sr,
        n_mels=n_mels,
        hop_length=hop_length,
        n_fft=n_fft,
        fmin=fmin,
        fmax=fmax,
        power=2.0
    )

    # Log-mel estable
    mel = np.log1p(mel).astype(np.float32)
    # z-score
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    return mel

## 2.3-Funcion principal
Aplica todas las funciones anteriores del apartado 2.2

In [ ]:
def preprocess_audio(filepath, noised=False):
    y, sr = load_audio(filepath)
    if noised:
        y = reduce_noise(y, sr)
    y = bandpass_filter(y, sr)
    y = normalize_audio(y)

    intervals = detect_active_segments(y, sr)

    mel_segments = []
    for start, end in intervals:
        segment = y[start:end]

        # si la duracion es menor a un segundo
        if (end - start) < sr*MIN_SEG_DURATION_SEC:
            continue  # descarta el segmento

        if is_likely_speech(segment, sr):
            continue

        mel = extract_mel(segment, sr, n_mels=64, hop_length=HOP_LENGTH)

        if mel is not None:
            mel_segments.append(mel)

    return mel_segments  # Lista de [128, fijo]

## 2.4-Aplicar preprocesado

In [ ]:
# Cargar taxonomy 
"""
taxonomy = pd.read_csv(path_taxonomy_csv, encoding='ISO-8859-1')
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str).str.strip()
valid_labels = set(taxonomy['primary_label'].unique())
print(f"[+] Especies taxonomy:{len(valid_labels)}")
label_to_index = {label: idx for idx, label in enumerate(valid_labels)}
"""
sample = pd.read_csv(path_sample_submission)
CLASSES = list(sample.columns[1:])  # orden oficial

label_to_index = {lab: i for i, lab in enumerate(CLASSES)}
index_to_label = {i: lab for lab, i in label_to_index.items()}

# preparar carpeta donde se guarda el preprocesado
#clean_folder(path_data)
mel_dir = create_folder(path_data, "mels")
print(mel_dir)

def get_files(path='/kaggle/input/birdclef-2025/test_soundscapes/'):
    if not os.path.isdir(path):
        return []
    test_soundscapes = [os.path.join(path, afile) for afile 
                    in sorted(os.listdir(path)) 
                    if afile.endswith('.ogg')]
    return test_soundscapes

In [ ]:
# Parametros
TEST_DIR   = Path(path_test_soundscapes)  
#TEST_DIR = Path(path_train_soundscapes)
NPZ_OUT    = Path(get_path(mel_dir, "test_mels.npz"))             
SR         = 32000
SEG_SEC    = 5                                                  # duración de cada trozo


# Buscar archivos
test_soundscapes = get_files()

# si no hay archivos de test, se cogen 700 de train_soundscapes
if len(test_soundscapes)==0:
    test_soundscapes = get_files(path=path_train_soundscapes)
    test_soundscapes = test_soundscapes[:1000]
    # coger 700 muestras al azar
    #test_soundscapes = random.sample(test_soundscapes, k=700)
    print(f"[+] No hay archivos encontrados en test, utilizando {len(test_soundscapes)} "
    f"archivos de train_soundscapes")
    TEST_DIR = Path(path_train_soundscapes)
    TEST_FLAG = False
else:
    print(f"[*] Utilizando {len(test_soundscapes)} de TEST")
    TEST_FLAG = True

# usa el mismo que en entrenamiento
def slice_one_minute_augmented(y, sr, seg=SEG_SEC):
    """Devuelve lista de segmentos de `seg` segundos (64 000 muestras cada 2 s)."""
    #samples = seg * sr
    #return [y[i*samples:(i+1)*samples] for i in range(60 // seg)]
    windows = []

    for end_time in range(5, 65, 5):
        official_start = end_time - 5
        official_end = end_time

        # Ventana oficial de 5 segundos
        offsets = [
            (official_start, official_end),      # 0-5
            (official_start, official_start+3),  # 0-3
            (official_start+1, official_start+4),# 1-4
            (official_start+2, official_end),    # 2-5
        ]

        for start_s, end_s in offsets:
            start_sample = int(start_s * sr)
            end_sample = int(end_s * sr)

            y_seg = y[start_sample:end_sample]

            # Importante: rellenar hasta 5 segundos
            y_seg = librosa.util.fix_length(y_seg, size=seg * sr)

            # Todas estas ventanas comparten el mismo end_time oficial
            windows.append((y_seg, end_time))

    return windows

def preprocess_segment_to_mel(y_seg, sr, 
                              apply_bandpass=APPLY_BANDPASS):
    #y_seg = reduce_noise(y_seg, sr)
    if apply_bandpass:
        y_seg = bandpass_filter(y_seg, sr)
    y_seg = normalize_audio(y_seg)

    mel = extract_mel(
        y_seg, sr, n_mels=N_MELS, 
        hop_length=HOP_LENGTH)
    return mel                                                     # shape [64, T] o None

In [ ]:
mel_segments_all  = []
segment_names_all = []

if TEST_DIR== path_train_soundscapes:
    wav_files=test_soundscapes
else:
    wav_files = sorted(TEST_DIR.glob('*.ogg'))
#i=0
print("[*] Procesando archivos")
for i,wav_path in enumerate(tqdm(wav_files, total=len(wav_files), desc='procesando test')):
    if i==10 and not TEST_FLAG:
        break
    # 3.1  cargamos el minuto completo
    y, sr = load_audio(wav_path, sr=SR)
    if len(y) < 59 * sr:            # por si algún archivo es mass corto
        y = librosa.util.fix_length(y, 60 * sr)

    # 3.2  lo partimos en bloques de 5 s y extraemos mel-spectrograma
    for seg, end_time in slice_one_minute_augmented(y, sr, seg=SEG_SEC):
        mel = preprocess_segment_to_mel(seg, sr)
        mel_segments_all.append(mel.astype(np.float16))

        # Aquí está la clave: varias ventanas tendrán el mismo row_id
        segment_names_all.append(f"{wav_path.stem}_{end_time}")


# guardado unico
np.savez_compressed(
    NPZ_OUT,
    mels = np.stack(mel_segments_all),   # [N, 64, T]
    ids  = np.array(segment_names_all)   # ['soundscape_12345_5', …]
)
print(f"[+] Listo: {len(segment_names_all)} segmentos guardados en {NPZ_OUT}")

# 3-Cargar modelo

## 3.1-Configuracion modelo

In [ ]:
# Configuración de modelos
NUM_CLASSES = 206
DROPOUT = 0.4
PRETRAINED = False  # En inferencia no descarga pesos; se cargan desde checkpoint
ENSEMBLE = False # True si se quiere hacer ensemble
ALPHA_ENSEMBLE = 0.35  # peso del primer modelo en ensemble
MODEL_NAME = "resnet" # prefijo del modelo cuando no aplica ensemble

if ENSEMBLE:
    MODEL_CONFIGS = [
        {
            "prefix": "effnet",
            "checkpoint": get_path("/kaggle/input/model-epochs/", "effnet_v3.0.pth"),
            "weight": ALPHA_ENSEMBLE,
        },
        {
            "prefix": "resnet",
            "checkpoint": get_path("/kaggle/input/model-epochs/", "resnet_v3.2.pth"),
            "weight": 1.0 - ALPHA_ENSEMBLE,
        },
    ]
else:
    MODEL_CONFIGS = [
        {
            "prefix": MODEL_NAME,
            "checkpoint": get_path("/kaggle/input/model-epochs/", "resnet_ep015.pth"),
            "weight": 1.0,
        }
    ]

# configuracion predicciones
POOL = "max"   # "mean"|"max"|"mix"|"topk"
ALPHA = 0.3 # solo afecta a 'mix'
TOPK  = 3

# configuracion inferencia
BATCH_SIZE = 8
device = "cpu"

## 3.2-Redes neuronales

### 3.2.1-BirdCNN

In [ ]:
class BirdCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.3, in_channels=1):  # <— 1 canal (log-Mel)
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),  # [B,16,128,T]
            nn.ReLU(),
            nn.MaxPool2d((3, 2)),                                  # [B,16,42,T/2]
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),                                  # [B,32,21,T/4]
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                           # [B,64,1,1]
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        if x.ndim == 3:                      # [B,128,T]
            x = x.unsqueeze(1)               # -> [B,1,128,T]
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

### 3.2.2-EfficientNet

In [ ]:
class BirdEfficientNet(nn.Module):
    def __init__(self, num_classes, in_chans=1, pretrained=False, dropout=0.3):
        super().__init__()
        # timm gestiona in_chans y el head; añadimos dropout con un head propio si quieres
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=pretrained,
            in_chans=in_chans,
            num_classes=0  # quitamos la head para añadir la nuestra con dropout
        )

        feat_dim = self.backbone.num_features
        
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feat_dim, num_classes)
        )

    def forward(self, x):
        if x.ndim == 3:
            x = x.unsqueeze(1)
        feats = self.backbone(x) # usa pesos pretrained
        logits = self.head(feats) # head aprende
        return logits

### 3.2.3-ResNet

In [ ]:
class BirdResNet(nn.Module):
    def __init__(self, num_classes, in_chans=1, pretrained=False, dropout=0.3):
        super().__init__()
        if pretrained:
            try:
                base = tv.resnet18(weights=tv.ResNet18_Weights.IMAGENET1K_V1)
            except Exception:
                base = tv.resnet18(pretrained=True)
        else:
            try:
                base = tv.resnet18(weights=None)
            except Exception:
                base = tv.resnet18(pretrained=False)

        # conv1 adapt
        orig = base.conv1
        if in_chans != orig.in_channels:
            new_conv = nn.Conv2d(in_chans, orig.out_channels,
                                 kernel_size=orig.kernel_size, stride=orig.stride,
                                 padding=orig.padding, bias=False)
            if pretrained:
                with torch.no_grad():
                    w = orig.weight.data
                    if in_chans == 1:
                        new_conv.weight.copy_(w.mean(1, keepdim=True))
                    else:
                        new_conv.weight.copy_(w.mean(1, keepdim=True).repeat(1, in_chans, 1, 1))
            base.conv1 = new_conv

        # backbone = todo menos la fc
        in_features = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base

        # head (puedes meter neck BN1d si quieres)
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        if x.ndim == 3:
            x = x.unsqueeze(1)
        feats = self.backbone(x)      # [B, in_features]
        logits = self.head(feats)
        return logits

## 3.3-Cargar modelo

In [ ]:
def build_model(model_prefix):
    if model_prefix == "cnn":
        model = BirdCNN(NUM_CLASSES, dropout=DROPOUT)
    elif model_prefix == "effnet":
        model = BirdEfficientNet(NUM_CLASSES, dropout=DROPOUT, pretrained=PRETRAINED)
    elif model_prefix == "resnet":
        model = BirdResNet(NUM_CLASSES, dropout=DROPOUT, pretrained=PRETRAINED)
    else:
        raise ValueError(f"Modelo no reconocido: {model_prefix}")
    
    return model

def load_model_from_config(cfg):
    model_prefix = cfg["prefix"]
    path_model = cfg["checkpoint"]

    model = build_model(model_prefix)

    ckpt = torch.load(path_model, map_location="cpu", weights_only=False)

    # El script de training guardó los pesos en la clave "model_state"
    state = ckpt["model_state"]

    model.load_state_dict(state)
    model = model.to(device)
    model.eval()

    print(f"[+] Modelo cargado correctamente: {model_prefix} desde {path_model}")

    return model

In [ ]:
for cfg in MODEL_CONFIGS:
    model = load_model_from_config(cfg)

# 4-Inferencia

## 4.0 funcion pooling

In [ ]:
def pool_preds(preds, mode="mix", alpha=0.5, topk=3):
    if preds.shape[0] == 1:
        return preds[0]
    if mode == "mean":
        return preds.mean(axis=0)
    if mode == "max":
        return preds.max(axis=0)
    if mode == "mix":
        return (1-alpha)*preds.mean(axis=0) + alpha*preds.max(axis=0)
    if mode == "topk":
        k = min(topk, preds.shape[0])
        topk_mean = np.sort(preds, axis=0)[-k:, :].mean(axis=0)
        return 0.5*preds.mean(axis=0) + 0.5*topk_mean
    raise ValueError(mode)

## 4.1-Cargar espectros

In [ ]:
# Cargar test npz
data = np.load(NPZ_OUT)

# 1) Extraer arrays: 'mels' tiene forma [N, N_MELS, T], y 'ids' son los row_id
mels_np = data["mels"]    # numpy array de shape (N, N_MELS, T)
row_ids = data["ids"]     # lista de strings (tamaño N)

N, n_mels, T = mels_np.shape
print(f"[+] {N} segmentos cargados (cada uno con {n_mels} bandas Mel y {T} frames)")

# Columnas de especies
df_sample = pd.read_csv(path_sample_submission)
species_cols = df_sample.columns[1:].tolist()
NUM_CLASSES = len(species_cols)

### 4.1.1- funcion auxiliar

In [ ]:
def predict_rows_single_model(model, mels_np, batch_size=BATCH_SIZE):
    """
    Devuelve probabilidades por fila/segmento del npz.
    Shape de salida: [N, NUM_CLASSES]
    """
    N = len(mels_np)
    probs_rows = np.zeros((N, NUM_CLASSES), dtype=np.float32)

    for start in tqdm(range(0, N, batch_size), desc="Predicciones desde .npz"):
        end = min(start + batch_size, N)

        batch_mels = mels_np[start:end]  # [B, MELS, T]
        batch = (
            torch.tensor(batch_mels, dtype=torch.float32)
            .unsqueeze(1)
            .to(device)
        )  # [B, 1, MELS, T]

        with torch.no_grad():
            logits = model(batch)
            probs = torch.sigmoid(logits).cpu().numpy()

        probs_rows[start:end] = probs

    return probs_rows

## 4.2-Generar predicciones

In [ ]:
ensemble_probs_rows = np.zeros((N, NUM_CLASSES), dtype=np.float32)
total_weight = 0.0

print("[*] Iniciando predicciones...")

for cfg in MODEL_CONFIGS:
    print(f"[*] Prediciendo con modelo: {cfg['prefix']} | peso={cfg['weight']}")

    model = load_model_from_config(cfg)

    probs_rows_model = predict_rows_single_model(
        model=model,
        mels_np=mels_np,
        batch_size=BATCH_SIZE
    )

    ensemble_probs_rows += cfg["weight"] * probs_rows_model
    total_weight += cfg["weight"]

    del model
    gc.collect()

ensemble_probs_rows /= total_weight

In [ ]:
# Pooling por row_id (solo si hay repetidos)

s = pd.Series(row_ids)

if len(s) == s.nunique():
    # no hay repetidos -> ya es final
    probs_all = ensemble_probs_rows
    row_ids_out = row_ids
else:
    agg = defaultdict(list)

    for rid, p in zip(row_ids, ensemble_probs_rows):
        agg[rid].append(p)

    row_ids_out = list(agg.keys())
    probs_all = np.zeros((len(row_ids_out), NUM_CLASSES), dtype=np.float32)

    for i, rid in enumerate(row_ids_out):
        preds = np.stack(agg[rid], axis=0)  # [n_windows, 206]
        probs_all[i] = pool_preds(preds, mode=POOL, alpha=ALPHA, topk=TOPK)

print("[INFO] probs_all shape:", probs_all.shape)

## 4.3-Generar submission.csv

In [ ]:
assert probs_all.shape[0] == len(row_ids_out), "El número de filas no coincide"
assert probs_all.shape[1] == len(species_cols), "El número de clases no coincide"

df_out = pd.DataFrame(probs_all, columns=species_cols)
df_out.insert(0, "row_id", row_ids_out)  # insertar primera columna row_id

# Guardar como CSV
df_out.to_csv("submission.csv", index=False)
print("[+] Predicciones a partir de .npz guardadas en 'submission.csv'")